# Benchmark Regression Review

Load benchmark scorecards and surface cases needing follow-up after nightly pipeline runs.

In [ ]:
from __future__ import annotations

import json
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
DEFAULT_SCORECARD = Path.cwd() / "benchmarks" / "results" / "scorecard.json"
scorecard_path = Path(os.environ.get("BENCHMARK_SCORECARD", str(DEFAULT_SCORECARD)))
if not scorecard_path.exists():
    raise FileNotFoundError(f"Scorecard not found at {scorecard_path}")

with scorecard_path.open('r', encoding='utf-8') as handle:
    raw = json.load(handle)

scorecard = pd.DataFrame.from_dict(raw, orient='index')
scorecard.index.name = 'case_id'
scorecard.reset_index(inplace=True)
scorecard.sort_values('total_duration_ms', ascending=False, inplace=True)
scorecard.head()

In [ ]:
failing = scorecard[scorecard['passed'] == False]
failing[['case_id', 'failed_assertions', 'total_duration_ms']]

In [ ]:
plt.figure(figsize=(10, 4))
plt.title('Benchmark Duration by Case (ms)')
plt.bar(scorecard['case_id'], scorecard['total_duration_ms'])
plt.xticks(rotation=45, ha='right')
plt.ylabel('Duration (ms)')
plt.tight_layout()
plt.show()

In [ ]:
summary = scorecard.groupby('passed').agg({
    'case_id': 'count',
    'tool_calls': 'sum',
    'turn_count': 'sum'
})
summary